# Notebook 01 — Production Structure Pipeline

Notebook này có một pipeline production duy nhất: tự resolve/restore Phase00, restore checkpoint theo `video_id + stage`, xử lý tuần tự từng video, package và sync Phase01. Chỉ sửa cell `USER SETTINGS`, sau đó **Run All**.

Business logic và cấu hình deterministic nằm trong `src/system1/` và `configs/`. Local Colab/Kaggle chỉ là scratch; checkpoint riêng tư trên Hugging Face là resume authority.

Repo được lấy từ GitHub bằng `git clone` lần đầu và `git fetch` + fast-forward ở lần sau (`git pull --ff-only` tương đương). Package được cài bằng `pip install -e` với extra production. Các root tương thích orchestration gồm `AIC_REPO_ROOT`, `AIC_REPO_PARENT`, `AIC_DATA_ROOT`, `AIC_RUNTIME_ROOT`, và `AIC_ARTIFACT_ROOT`.

In [ ]:
# USER SETTINGS — chỉ chỉnh các giá trị phụ thuộc người chạy/môi trường.
batch_id = "batch_000"
worker_id = "worker_000"
release_id_override = None  # None = auto-resolve theo completed_at; Phase00 v001 cũ thiếu field này thì đặt "canonical_release_v001"

# Để None để dùng repo/versioned default trong configs/storage.yaml.
hf_release_repo = None
hf_checkpoint_repo = None  # repo checkpoint phải private
hf_revision = None
hf_prefix = None
checkpoint_revision = None
checkpoint_prefix = None
scratch_dir_override = None

github_repo_url = "https://github.com/awun0105/Multimodal-Agentic-Retrieval-Engine.git"
github_branch = "system1-notebook01"
repo_dir_name = "Multimodal-Agentic-Retrieval-Engine"

In [ ]:
# BƯỚC 1: Detect environment, paths và secrets. Không in secret.
import os, sys
from pathlib import Path

if "google.colab" in sys.modules:
    runtime_env = "colab"
    workspace = Path("/content/aic_phase01")
elif "KAGGLE_URL_BASE" in os.environ or "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    runtime_env = "kaggle"
    workspace = Path("/kaggle/working/aic_phase01")
else:
    runtime_env = "local"
    workspace = Path.cwd() / ".phase01_runtime"
workspace.mkdir(parents=True, exist_ok=True)
output_root = workspace / "output"
scratch_dir = Path(scratch_dir_override).expanduser() if scratch_dir_override else workspace / "scratch"
model_cache = workspace / "model_cache"
for path in (output_root, scratch_dir, model_cache): path.mkdir(parents=True, exist_ok=True)

def load_secret(name):
    if os.environ.get(name): return os.environ[name]
    if runtime_env == "colab":
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:
            return None
    if runtime_env == "kaggle":
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        except Exception:
            return None
    return None

for secret_name in ("HF_TOKEN", "GEMINI_API_KEY"):
    value = load_secret(secret_name)
    if value: os.environ[secret_name] = value
if os.environ.get("HF_TOKEN"): os.environ["AIC_HF_TOKEN"] = os.environ["HF_TOKEN"]
if not os.environ.get("HF_TOKEN") or not os.environ.get("GEMINI_API_KEY"):
    raise RuntimeError("Cần cấu hình HF_TOKEN và GEMINI_API_KEY trong Colab/Kaggle Secrets hoặc environment.")

os.environ["HF_HOME"] = str(model_cache / "hf")
os.environ["AIC_DATA_ROOT"] = str(workspace / "data")
os.environ["AIC_RUNTIME_ROOT"] = str(workspace / "runtime")
os.environ["AIC_ARTIFACT_ROOT"] = str(workspace / "artifacts")
print({"environment": runtime_env, "workspace": str(workspace), "output_root": str(output_root), "scratch": str(scratch_dir)})

In [ ]:
# BƯỚC 2: Clone/update đúng branch GitHub mà không xóa thay đổi local.
import subprocess

def run_command(command, cwd=None):
    print("RUN:", " ".join(map(str, command)))
    result = subprocess.run(command, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    if result.returncode: raise RuntimeError(f"Command failed ({result.returncode}): {command}")
    return result

def is_repo_root(path): return (path / "system1" / "src" / "system1" / "__init__.py").is_file()
repo_root = next((path for path in [Path.cwd(), *Path.cwd().parents] if is_repo_root(path)), None)
if repo_root is None:
    repo_root = workspace / repo_dir_name
    if not (repo_root / ".git").exists():
        run_command(["git", "clone", "--branch", github_branch, "--single-branch", github_repo_url, str(repo_root)], cwd=workspace)
if run_command(["git", "status", "--porcelain"], cwd=repo_root).stdout.strip():
    raise RuntimeError("Repo có thay đổi local; notebook không tự reset/stash.")
run_command(["git", "fetch", "origin", github_branch], cwd=repo_root)
local_branch = subprocess.run(["git", "show-ref", "--verify", "--quiet", f"refs/heads/{github_branch}"], cwd=repo_root).returncode == 0
run_command(["git", "switch", github_branch] if local_branch else ["git", "switch", "--track", "-c", github_branch, f"origin/{github_branch}"], cwd=repo_root)
run_command(["git", "merge", "--ff-only", f"origin/{github_branch}"], cwd=repo_root)
os.environ["AIC_REPO_ROOT"] = str(repo_root)
os.environ["AIC_REPO_PARENT"] = str(repo_root.parent)

In [ ]:
# BƯỚC 3: Cài production dependencies và xác nhận kernel import đúng source.
system1_root = repo_root / "system1"
run_command([sys.executable, "-m", "pip", "install", "-q", "-e", f"{system1_root}[phase01-production]"])
source_root = str(system1_root / "src")
sys.path = [source_root] + [item for item in sys.path if item != source_root]
for name in list(sys.modules):
    if name == "system1" or name.startswith("system1."): del sys.modules[name]
import system1
actual = Path(system1.__file__).resolve()
expected = (system1_root / "src" / "system1" / "__init__.py").resolve()
if actual != expected: raise RuntimeError(f"Import source mismatch: actual={actual}, expected={expected}")
print("package source preflight: OK", actual)

In [ ]:
# BƯỚC 4: Helper CLI có streaming output và error tail.
def run_cli(arguments):
    command = [sys.executable, "-m", "system1.cli", *map(str, arguments)]
    env = os.environ.copy(); env["PYTHONUNBUFFERED"] = "1"
    process = subprocess.Popen(command, cwd=system1_root, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
    lines = []
    for line in process.stdout:
        print(line, end="", flush=True); lines.append(line)
    code = process.wait()
    if code:
        raise RuntimeError(f"CLI failed ({code}). Last output:\n" + "".join(lines[-120:]))
    return "".join(lines)

In [ ]:
# BƯỚC 5: Một lệnh production — auto-resolve, restore, preflight, resume, package, sync.
command = [
    "process-batch", "--batch-id", batch_id, "--worker-id", worker_id,
    "--output", str(output_root), "--scratch-dir", str(scratch_dir),
    "--require-frame-timeline", "--restore-phase00", "--validate-remote", "--sync",
]
optional = {
    "--release-id-override": release_id_override,
    "--hf-repo-id": hf_release_repo,
    "--hf-checkpoint-repo": hf_checkpoint_repo,
    "--hf-revision": hf_revision,
    "--hf-prefix": hf_prefix,
    "--checkpoint-revision": checkpoint_revision,
    "--checkpoint-prefix": checkpoint_prefix,
}
for option, value in optional.items():
    if value not in (None, ""): command.extend([option, str(value)])
run_cli(command)

In [ ]:
# BƯỚC 6: Báo cáo ngắn sau Run All.
import json
last_run = json.loads((output_root / "phase01_last_run.json").read_text())
release_root = Path(last_run["release_dir"])
resolved = json.loads((release_root / "manifests/phase01/resolved_config.json").read_text())
report_path = release_root / "manifests/worker_reports" / f"structure_{batch_id}_{worker_id}.json"
review_path = release_root / "manifests/phase01" / f"manual_review_{batch_id}_{worker_id}.json"
report = json.loads(report_path.read_text())
review = json.loads(review_path.read_text())
print({
    "release_id": resolved["runtime"]["release_id"],
    "config_hash": resolved["config_hash"],
    "counts": report.get("counts"),
    "manual_review_status": review["status"],
    "manual_review_samples": review["sample_size_actual"],
    "report": str(report_path),
})